# YOLO 2회차 실습 A — 컨베이어 상자 '크기별' 자동 집계

택배 분류장 컨베이어 CCTV 영상에서 **지나간 상자를 소형·중형·대형(S/M/L)으로 나눠 세는 기능**을 처음부터 만들어 봅니다.
단순히 개수만 세는 것이 아니라, 세그멘테이션 마스크로 상자의 실제 크기를 재고 크기 기준을 데이터로 정하는 데까지 갑니다.
각 단계는 앞 단계의 결과물을 그대로 이어 받아 기능을 하나씩 더합니다.

| 단계 | 만드는 것 | 결과물 |
|---|---|---|
| Step 1 | 사전학습 모델 그대로 써 보기 — 무엇을 못 하는지 확인 | 탐지 결과 비교 |
| Step 2 | 세그멘테이션 모델 파인튜닝 + 성능 평가 (박스·마스크 mAP, 임계값, 오류 분석) | `box_seg_best.pt`, 평가 지표 표 |
| Step 3 | 트래킹 — 프레임이 바뀌어도 같은 상자에 같은 ID 붙이기 | `tracked.mp4` |
| Step 4 | 라인 카운팅 — 기준선을 넘은 상자 수 세기 | `counting.mp4`, `box_count_log.csv` |
| Step 5 | 크기 분류 — 마스크로 크기를 재고, S/M/L 기준을 데이터로 정해 크기별로 세기 | `size_counting.mp4`, `box_size_log.csv`, `size_rules.json` |

**데이터셋**: Ultralytics에서 공개한 *Package Segmentation* 데이터셋 (택배 분류장 CCTV 이미지, 클래스 1개 `package`, 학습 1,920장 · 검증 188장 · 테스트 89장, 약 100MB)

**실행 환경**: Google Colab, 런타임 유형 **T4 GPU** (메뉴 → 런타임 → 런타임 유형 변경)

> 결과 파일은 Google Drive의 `MyDrive/yolo_day2` 폴더에 저장됩니다. Colab 런타임이 끊겨도 학습한 모델이 남고, 마지막 통합 실습(`yolo_day2_monitor_lab`)에서 이 파일들을 다시 사용합니다.

## 0. 준비

In [ ]:
!pip -q install ultralytics lap shapely

In [ ]:
import os, json, shutil, base64, subprocess, re, time, glob
from pathlib import Path
from collections import Counter, defaultdict

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import HTML, Image, display
from ultralytics import YOLO

print('GPU 사용 가능:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# 결과 저장 위치 — Google Drive를 연결하면 런타임이 끊겨도 결과가 남습니다
USE_DRIVE = True

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and USE_DRIVE:
    drive.mount('/content/drive')            # 처음 한 번 Google 계정 접근 허용 창이 뜹니다
    SAVE_DIR = Path('/content/drive/MyDrive/yolo_day2')
else:
    SAVE_DIR = Path('outputs')
SAVE_DIR.mkdir(parents=True, exist_ok=True)
print('결과 저장 위치:', SAVE_DIR.resolve())

In [ ]:
# 학습 설정 — 시간이 부족하면 QUICK_RUN = True (학습을 짧게 줄입니다)
QUICK_RUN = False

CFG = dict(epochs=20, imgsz=640, batch=16) if not QUICK_RUN else dict(epochs=3, imgsz=416, batch=16)
print(CFG)

In [ ]:
# 화면 표시용 도우미 함수
def show_bgr(img, title=None, size=(9, 6)):
    plt.figure(figsize=size)
    plt.imshow(img[..., ::-1]); plt.axis('off')
    if title:
        plt.title(title)
    plt.show()

def show_grid(imgs, titles=None, cols=2, size=4.5):
    rows = (len(imgs) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(size * cols, size * rows))
    for ax, i in zip(np.array(axes).ravel(), range(rows * cols)):
        ax.axis('off')
        if i < len(imgs):
            ax.imshow(imgs[i][..., ::-1])
            if titles:
                ax.set_title(titles[i], fontsize=9)
    plt.tight_layout(); plt.show()

def show_video(path, width=360):
    """mp4 영상을 브라우저에서 재생 가능한 H.264로 바꿔 노트북에 표시"""
    h264 = str(Path(path).with_name(Path(path).stem + '_h264.mp4'))
    subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', str(path),
                    '-vcodec', 'libx264', '-pix_fmt', 'yuv420p', h264], check=True)
    data = base64.b64encode(open(h264, 'rb').read()).decode()
    display(HTML(f'<video width="{width}" controls loop autoplay muted>'
                 f'<source src="data:video/mp4;base64,{data}" type="video/mp4"></video>'))

def save_out(path):
    """결과 파일을 저장 폴더(SAVE_DIR)에 복사"""
    dst = SAVE_DIR / Path(path).name
    if Path(path).resolve() != dst.resolve():
        shutil.copy(path, dst)
    return dst

---
## Step 1. 사전학습 모델 그대로 써 보기

### 1-1. 데이터셋 내려받기와 라벨 구조 확인

`check_det_dataset()`은 데이터셋 설정 파일(`package-seg.yaml`)을 읽고, 데이터가 없으면 자동으로 내려받습니다.

In [ ]:
from ultralytics.data.utils import check_det_dataset

data = check_det_dataset('package-seg.yaml')
DATA_ROOT = Path(data['path'])
print('데이터 위치:', DATA_ROOT)
print('클래스:', data['names'])
for split in ['train', 'val', 'test']:
    n_img = len(list((DATA_ROOT / 'images' / split).glob('*.jpg')))
    n_lbl = len(list((DATA_ROOT / 'labels' / split).glob('*.txt')))
    print(f'{split:5s}: 이미지 {n_img:5d}장 / 라벨 {n_lbl:5d}개')

라벨 파일 한 줄은 상자 하나입니다. 이 데이터셋은 상자 테두리를 **다각형 꼭짓점 좌표**로 적어 두었습니다. 사각형 박스보다 정보가 많아서, 뒤에서 상자의 실제 크기를 잴 때 이 정보를 씁니다.

```
클래스번호  x1 y1  x2 y2  x3 y3 ...   ← 모두 0~1로 정규화된 좌표
```

In [ ]:
def read_gt(img_path):
    """이미지 경로 → (정답 박스 (N,4) [x1,y1,x2,y2], 정답 다각형 목록) 픽셀 좌표"""
    img_path = Path(img_path)
    lbl = img_path.parent.parent.parent / 'labels' / img_path.parent.name / (img_path.stem + '.txt')
    h, w = cv2.imread(str(img_path)).shape[:2]
    boxes, polys = [], []
    if lbl.exists():
        for line in lbl.read_text().strip().splitlines():
            v = np.array(line.split()[1:], float)
            xs, ys = v[0::2] * w, v[1::2] * h
            polys.append(np.stack([xs, ys], 1))
            boxes.append([xs.min(), ys.min(), xs.max(), ys.max()])
    return np.array(boxes).reshape(-1, 4), polys

def draw_boxes(img, boxes, color, label=None, thick=2):
    for b in boxes:
        x1, y1, x2, y2 = map(int, b[:4])
        cv2.rectangle(img, (x1, y1), (x2, y2), color, thick)
        if label:
            cv2.putText(img, label, (x1, max(15, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
    return img

def draw_polys(img, polys, color, alpha=0.35):
    overlay = img.copy()
    for p in polys:
        cv2.fillPoly(overlay, [p.astype(np.int32)], color)
        cv2.polylines(img, [p.astype(np.int32)], True, color, 2)
    return cv2.addWeighted(overlay, alpha, img, 1 - alpha, 0)

VAL_IMAGES = sorted((DATA_ROOT / 'images' / 'val').glob('*.jpg'))
SAMPLES = [VAL_IMAGES[i] for i in (3, 40, 95, 150)]

gt_imgs = []
for p in SAMPLES:
    b, poly = read_gt(p)
    gt_imgs.append(draw_boxes(draw_polys(cv2.imread(str(p)), poly, (0, 255, 0)), b, (0, 200, 255), thick=1))
show_grid(gt_imgs, [f'GT: {len(read_gt(p)[1])} packages (polygon + box)' for p in SAMPLES])

초록색이 다각형 정답, 주황색 얇은 선이 다각형을 감싸는 사각형입니다. **비스듬히 놓인 상자는 사각형이 실제 상자보다 훨씬 크게** 잡히는 것을 확인하세요.

### 1-2. COCO 사전학습 모델로 탐지해 보기

`yolo26n.pt`는 COCO 데이터셋(사람·자동차·가방 등 80개 클래스)으로 학습된 모델입니다. 택배 상자도 찾을 수 있을까요?

In [ ]:
coco_model = YOLO('yolo26n.pt')

pred_imgs = [coco_model(str(p), verbose=False, conf=0.25)[0].plot() for p in SAMPLES]
show_grid(pred_imgs, ['COCO pretrained'] * len(SAMPLES))

In [ ]:
# 검증 이미지 전체에서 어떤 클래스로 탐지하는지 세어 봅니다
cls_counter, no_det, total_gt = Counter(), 0, 0
for p in VAL_IMAGES:
    r = coco_model(str(p), verbose=False, conf=0.25)[0]
    names = [r.names[int(c)] for c in r.boxes.cls]
    cls_counter.update(names)
    no_det += (len(names) == 0)
    total_gt += len(read_gt(p)[0])

print(f'검증 이미지 {len(VAL_IMAGES)}장, 정답 상자 {total_gt}개')
print(f'아무것도 찾지 못한 이미지: {no_det}장')
pd.DataFrame(cls_counter.most_common(10), columns=['coco_class', 'count'])

COCO에는 '택배 상자' 클래스가 없습니다. 그래서 상자는 거의 찾지 못하고, 긴 컨베이어 벨트를 **`train`(기차)** 으로 착각하는 일이 많습니다.
'물체처럼 보이는 것'을 찾는 능력은 이미 갖고 있으니, **우리 대상의 이름과 모양**만 우리 데이터로 가르치면 됩니다. 이것이 Step 2의 파인튜닝입니다.

---
## Step 2. 세그멘테이션 모델 파인튜닝과 성능 평가

### 2-1. 왜 세그멘테이션 모델인가

탐지(detect) 모델은 사각형 박스만 돌려줍니다. 세그멘테이션(segment) 모델은 박스와 함께 **물체 모양 그대로의 마스크**를 돌려줍니다.
상자 크기를 재려면 마스크가 필요합니다. 비스듬한 상자의 박스 면적은 실제보다 크게 나오기 때문입니다.

| 모델 | 가중치 | 출력 | 이번 실습에서 쓰는 곳 |
|---|---|---|---|
| 탐지 | `yolo26n.pt` | 박스 | Step 1 비교용 |
| 세그멘테이션 | `yolo26n-seg.pt` | 박스 + 마스크 | Step 2~5 전체 |

### 2-2. 학습

사전학습 가중치(`yolo26n-seg.pt`)에서 출발해 상자 데이터로 추가 학습합니다. T4 GPU 기준 20 epoch에 15~20분 정도 걸립니다.

In [ ]:
model = YOLO('yolo26n-seg.pt')
t0 = time.time()
model.train(data='package-seg.yaml', epochs=CFG['epochs'], imgsz=CFG['imgsz'], batch=CFG['batch'],
            project='runs_box', name='seg_baseline', exist_ok=True, seed=0, verbose=False)
TRAIN_MIN = (time.time() - t0) / 60
RUN_DIR = Path(model.trainer.save_dir)
BEST = RUN_DIR / 'weights' / 'best.pt'
print(f'학습 시간: {TRAIN_MIN:.1f}분')
print('최고 성능 가중치:', BEST)
shutil.copy(BEST, SAVE_DIR / 'box_seg_best.pt')
print('저장 폴더에 복사:', SAVE_DIR / 'box_seg_best.pt')

In [ ]:
# epoch별 손실과 지표 변화
display(Image(filename=str(RUN_DIR / 'results.png'), width=900))

그래프 읽는 법
- **train/val loss**가 함께 내려가면 정상적으로 배우는 중입니다. train만 내려가고 val이 올라가면 과적합입니다.
- 세그멘테이션 모델은 손실이 하나 더 있습니다. **seg_loss**는 마스크 모양이 얼마나 정답과 다른지를 나타냅니다.
- **metrics/mAP50(B)** 는 박스, **metrics/mAP50(M)** 은 마스크 기준 점수입니다.

### 2-3. 성능 지표 확인

| 지표 | 질문 | 계산 |
|---|---|---|
| Precision (정밀도) | 상자라고 한 것 중 진짜 상자는 몇 %? | TP / (TP + FP) |
| Recall (재현율) | 진짜 상자 중 찾아낸 것은 몇 %? | TP / (TP + FN) |
| mAP50 | IoU 0.5 기준으로 본 종합 점수 | 정밀도-재현율 곡선 아래 면적 |
| mAP50-95 | 위치·모양까지 얼마나 정확한가 | IoU 0.5~0.95 구간 평균 |

IoU(Intersection over Union)는 예측과 정답이 겹치는 비율입니다. 박스 지표는 사각형끼리, **마스크 지표는 픽셀 단위 모양끼리** 겹침을 계산하므로 마스크 점수가 더 엄격합니다.

In [ ]:
best_model = YOLO(BEST)

def evaluate(weights_model, split):
    m = weights_model.val(data='package-seg.yaml', split=split, imgsz=CFG['imgsz'], batch=CFG['batch'],
                          project='runs_box', name=f'val_{split}', exist_ok=True, verbose=False)
    return m, {'split': split,
               'box_precision': round(m.box.mp, 3), 'box_recall': round(m.box.mr, 3),
               'box_map50': round(m.box.map50, 3), 'box_map50_95': round(m.box.map, 3),
               'mask_map50': round(m.seg.map50, 3), 'mask_map50_95': round(m.seg.map, 3)}

val_metrics, val_row = evaluate(best_model, 'val')
test_metrics, test_row = evaluate(best_model, 'test')
metrics_table = pd.DataFrame([val_row, test_row])
metrics_table

학습에 쓰지 않은 **test** 세트 점수가 val과 비슷하면, 모델이 특정 이미지만 외운 것이 아니라 일반화되었다는 뜻입니다.

In [ ]:
VAL_DIR = Path(val_metrics.save_dir)
for name in ['BoxPR_curve.png', 'MaskPR_curve.png']:
    if (VAL_DIR / name).exists():
        display(Image(filename=str(VAL_DIR / name), width=480))

### 2-4. 박스와 마스크, 면적이 얼마나 다른가

검증 이미지의 모든 예측에 대해 **박스 면적 ÷ 마스크 면적** 을 계산합니다. 1에 가까우면 박스가 상자를 꼭 맞게 감싼 것이고, 클수록 상자가 비스듬해 박스에 빈 공간이 많은 것입니다.

In [ ]:
ratios, worst = [], None
for p in VAL_IMAGES:
    r = best_model(str(p), conf=0.4, imgsz=CFG['imgsz'], verbose=False)[0]
    if r.masks is None:
        continue
    for b, poly in zip(r.boxes.xyxy.cpu().numpy(), r.masks.xy):
        if len(poly) < 3:
            continue
        mask_area = cv2.contourArea(poly.astype(np.float32))
        box_area = (b[2] - b[0]) * (b[3] - b[1])
        if mask_area > 500:
            ratio = box_area / mask_area
            ratios.append(ratio)
            if worst is None or ratio > worst[0]:
                worst = (ratio, p, b, poly)

ratios = np.array(ratios)
print(f'예측 {len(ratios)}개 | 박스/마스크 면적비 중앙값 {np.median(ratios):.2f} | '
      f'1.5배 이상인 예측 {np.mean(ratios >= 1.5) * 100:.0f}%')

ratio, p, b, poly = worst
im = draw_polys(cv2.imread(str(p)), [poly], (0, 255, 0))
draw_boxes(im, [b], (0, 0, 255), thick=2)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(ratios, bins=30, color='tab:blue'); axes[0].axvline(1.5, ls='--', c='gray')
axes[0].set_xlabel('box area / mask area'); axes[0].set_ylabel('count')
axes[1].imshow(im[..., ::-1]); axes[1].axis('off')
axes[1].set_title(f'largest gap: box is {ratio:.1f}x the mask (red = box, green = mask)', fontsize=9)
plt.tight_layout(); plt.show()

박스 면적으로 크기를 나누면, 같은 상자라도 **놓인 각도에 따라 S가 되었다가 L이 되었다가** 합니다. 그래서 Step 5에서는 마스크 면적을 씁니다.

### 2-5. 신뢰도 임계값 정하기 — 직접 TP/FP/FN 세어 보기

mAP는 모든 임계값을 종합한 점수입니다. 하지만 실제 서비스에서는 **임계값(conf) 하나를 골라야** 합니다.
검증 이미지마다 예측을 한 번만 해 두고, 임계값을 바꿔 가며 TP·FP·FN을 직접 세어 봅니다.

In [ ]:
def box_iou(a, b):
    """a: (N,4), b: (M,4) → (N,M) IoU 행렬"""
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1]); area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / (area_a[:, None] + area_b[None, :] - inter + 1e-9)

def match(pred, gt, iou_thr=0.5):
    """신뢰도 높은 예측부터 정답과 1:1로 짝짓기 → (TP, FP, FN, 맞은 예측 여부)"""
    order = np.argsort(-pred[:, 4]) if len(pred) else []
    ious = box_iou(pred[:, :4], gt) if len(pred) else np.zeros((0, len(gt)))
    used, hit = set(), np.zeros(len(pred), bool)
    for i in order:
        if len(gt) == 0:
            break
        j = int(np.argmax(np.where([k in used for k in range(len(gt))], -1, ious[i])))
        if ious[i, j] >= iou_thr and j not in used:
            used.add(j); hit[i] = True
    tp = int(hit.sum())
    return tp, len(pred) - tp, len(gt) - tp, hit

# 예측은 낮은 임계값(0.05)으로 한 번만 수행해 저장
cache = []
for p in VAL_IMAGES:
    r = best_model(str(p), conf=0.05, imgsz=CFG['imgsz'], verbose=False)[0]
    pred = np.hstack([r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy()[:, None]])
    cache.append((p, pred, read_gt(p)[0]))

rows = []
for thr in [0.1, 0.25, 0.4, 0.55, 0.7, 0.85]:
    tp = fp = fn = 0
    for _, pred, gt in cache:
        t, f, n, _ = match(pred[pred[:, 4] >= thr], gt)
        tp += t; fp += f; fn += n
    prec, rec = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
    rows.append({'conf': thr, 'tp': tp, 'fp': fp, 'fn': fn, 'precision': round(prec, 3),
                 'recall': round(rec, 3), 'f1': round(2 * prec * rec / max(prec + rec, 1e-9), 3)})
thr_table = pd.DataFrame(rows)
BEST_CONF = float(thr_table.loc[thr_table['f1'].idxmax(), 'conf'])
print('F1이 가장 높은 임계값:', BEST_CONF)
thr_table

In [ ]:
plt.figure(figsize=(7, 3.5))
plt.plot(thr_table['conf'], thr_table['precision'], '-o', label='precision')
plt.plot(thr_table['conf'], thr_table['recall'], '-o', label='recall')
plt.plot(thr_table['conf'], thr_table['f1'], '-o', label='f1')
plt.axvline(BEST_CONF, ls='--', c='gray')
plt.xlabel('confidence threshold'); plt.ylim(0, 1.05); plt.grid(alpha=.3); plt.legend(); plt.show()

임계값을 올리면 FP(헛탐지)가 줄어 정밀도가 오르지만, FN(놓침)이 늘어 재현율이 떨어집니다.
**카운팅 서비스**라면 상자를 놓치는 것도, 없는 상자를 세는 것도 모두 숫자를 틀리게 만들므로 둘의 균형인 F1이 가장 높은 값을 출발점으로 삼습니다.

### 2-6. 오류 분석 — 어디서 틀리는가

틀린 개수(FP + FN)가 가장 많은 이미지를 골라 직접 봅니다. 초록 = 정답, 파랑 = 맞은 예측, 빨강 = 틀린 예측(FP)입니다.

In [ ]:
err_rows = []
for p, pred, gt in cache:
    pr = pred[pred[:, 4] >= BEST_CONF]
    tp, fp, fn, hit = match(pr, gt)
    err_rows.append((fp + fn, fp, fn, p, pr, gt, hit))
err_rows.sort(key=lambda x: -x[0])

imgs, titles = [], []
for err, fp, fn, p, pr, gt, hit in err_rows[:4]:
    im = cv2.imread(str(p))
    draw_boxes(im, gt, (0, 255, 0), thick=3)
    draw_boxes(im, pr[hit], (255, 128, 0))
    draw_boxes(im, pr[~hit], (0, 0, 255), 'FP')
    imgs.append(im); titles.append(f'FP {fp} / FN {fn} | {p.name[:28]}')
show_grid(imgs, titles)
print('오류가 하나도 없는 이미지 비율:', round(np.mean([e[0] == 0 for e in err_rows]), 3))

**확인해 보기** — 틀린 이미지들에서 공통점을 찾아 아래처럼 분류해 보세요. 개선 방향이 원인마다 다릅니다.

| 원인 유형 | 예시 | 개선 방향 |
|---|---|---|
| 겹침·밀집 | 상자가 겹쳐 하나로 잡힘 | 해상도(imgsz) 올리기, 겹친 장면 데이터 추가 |
| 화면 가장자리 잘림 | 일부만 보이는 상자 | 라벨 기준 통일, 집계 영역에서 가장자리 제외 |
| 비슷한 물체 | 비닐 포대·작업자 손 | 헷갈리는 사례를 라벨링해 추가 |
| 라벨 누락 | 정답에 없는데 모델이 찾은 상자 | 라벨 검수 (모델이 맞는 경우도 있음) |

### 2-7. 개선 실험 (선택)

한 번에 하나만 바꿔서 비교해야 무엇이 효과가 있었는지 알 수 있습니다. `RUN_EXPERIMENT = True`로 바꾸면 두 가지 개선안을 추가로 학습해 기준 모델과 비교합니다 (T4 기준 약 40분 추가).

In [ ]:
RUN_EXPERIMENT = False

EXPERIMENTS = [
    {'exp_name': 'more_epochs', 'model': 'yolo26n-seg.pt', 'epochs': CFG['epochs'] * 2, 'imgsz': CFG['imgsz']},
    {'exp_name': 'bigger_model', 'model': 'yolo26s-seg.pt', 'epochs': CFG['epochs'], 'imgsz': CFG['imgsz']},
]
METRIC_KEYS = ['box_map50', 'box_map50_95', 'mask_map50', 'mask_map50_95']

exp_rows = [{'exp_name': 'baseline', 'model': 'yolo26n-seg.pt', 'epochs': CFG['epochs'], 'imgsz': CFG['imgsz'],
             **{k: val_row[k] for k in METRIC_KEYS}, 'train_min': round(TRAIN_MIN, 1)}]

if RUN_EXPERIMENT:
    for e in EXPERIMENTS:
        m = YOLO(e['model']); t0 = time.time()
        m.train(data='package-seg.yaml', epochs=e['epochs'], imgsz=e['imgsz'], batch=CFG['batch'],
                project='runs_box', name=e['exp_name'], exist_ok=True, seed=0, verbose=False)
        mins = (time.time() - t0) / 60
        _, row = evaluate(YOLO(Path(m.trainer.save_dir) / 'weights' / 'best.pt'), 'val')
        exp_rows.append({**e, **{k: row[k] for k in METRIC_KEYS}, 'train_min': round(mins, 1)})

exp_table = pd.DataFrame(exp_rows)
exp_table.to_csv('experiments.csv', index=False); save_out('experiments.csv')
exp_table

개선안이 기준 모델보다 좋아졌더라도 **학습 시간·추론 속도**가 함께 늘어나는지 확인해야 합니다. 3회차에서 다룰 엣지 장비(Jetson)에서는 작은 모델의 속도가 더 중요할 수 있습니다.

---
## Step 3. 트래킹 — 같은 상자에 같은 ID 붙이기

### 3-1. 실습용 컨베이어 영상 만들기

이 데이터셋에는 같은 CCTV에서 연속으로 뽑은 프레임(`frame_1` ~ `frame_307`)이 들어 있습니다. 이 프레임들을 순서대로 이어 붙여 약 17초짜리 컨베이어 영상을 만듭니다.
CCTV 화면은 **좌우 두 화면이 한 장에 붙어 있는 형태**라, 상자가 왼쪽에서 오른쪽으로 흐르는 **오른쪽 화면만 잘라** 씁니다.
학습 세트의 이미지에는 밝기·회전 등 변형(augmentation)을 준 사본이 3장씩 있으므로, 바로 앞 프레임과 가장 비슷한 사본을 골라 자연스럽게 잇습니다.

> 이 영상에는 학습에 쓰인 프레임이 섞여 있습니다. 그래서 여기서는 **정확도 평가가 아니라 트래킹·집계 기능을 구현하는 연습용**으로 씁니다. 정확도 평가는 Step 2의 val/test 결과를 기준으로 합니다.

In [ ]:
def build_conveyor_clip(root, out='conveyor_clip.mp4', first=1, last=307, fps=15, panel='right'):
    by_num = defaultdict(list)
    for f in glob.glob(str(root / 'images' / '*' / 'frame_*_jpg*')):
        n = int(re.search(r'frame_(\d+)_jpg', f).group(1))
        if first <= n <= last:
            by_num[n].append(f)
    nums = sorted(by_num)
    load = lambda f: cv2.imread(f).astype(np.float32)
    anchor = next(n for n in nums if '/train/' not in by_num[n][0])   # 변형이 없는 val/test 프레임에서 출발
    chosen = {anchor: by_num[anchor][0]}
    a = nums.index(anchor)
    for seq in (nums[a + 1:], nums[:a][::-1]):                         # 앞쪽·뒤쪽으로 이어 가며 선택
        prev = load(chosen[anchor])
        for n in seq:
            diffs = [np.abs(load(c) - prev).mean() for c in by_num[n]]
            chosen[n] = by_num[n][int(np.argmin(diffs))]
            prev = load(chosen[n])
    crop = lambda im: im[:, im.shape[1] // 2:] if panel == 'right' else im[:, :im.shape[1] // 2]
    h, w = crop(cv2.imread(chosen[nums[0]])).shape[:2]
    writer = cv2.VideoWriter(out, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))
    for n in nums:
        writer.write(np.ascontiguousarray(crop(cv2.imread(chosen[n]))))
    writer.release()
    return out, len(nums), fps, (w, h)

CLIP, n_frames, CLIP_FPS, (W, H) = build_conveyor_clip(DATA_ROOT)
save_out(CLIP)
print(f'{CLIP}: {n_frames}프레임, {CLIP_FPS}fps, {W}x{H}, 약 {n_frames / CLIP_FPS:.1f}초')
show_video(CLIP)

### 3-2. 탐지만으로는 개수를 셀 수 없는 이유

프레임마다 탐지된 상자 수를 모두 더하면 '실제로 지나간 상자 수'와 전혀 다른 숫자가 나옵니다. 같은 상자가 수십 프레임에 걸쳐 반복해서 잡히기 때문입니다.

In [ ]:
per_frame = [len(r.boxes) for r in best_model(CLIP, stream=True, conf=BEST_CONF, imgsz=CFG['imgsz'], verbose=False)]
print('프레임별 탐지 수 합계:', sum(per_frame), '| 프레임당 평균:', round(np.mean(per_frame), 2))

### 3-3. 트래킹 적용

트래커는 이전 프레임의 박스 위치와 움직임을 기억해 두었다가, 새 프레임의 탐지 결과와 짝을 지어 **같은 물체에 같은 ID**를 붙입니다.
여기서는 빠르고 널리 쓰이는 **ByteTrack**(`bytetrack.yaml`)을 씁니다.

> 영상 파일 전체를 `track(source=영상, stream=True)`로 한 번에 넘기면 트래커가 영상 시작부터 끝까지 상태를 알아서 이어 갑니다. 반대로 카메라 프레임을 한 장씩 꺼내 `track(frame, persist=True)`를 반복 호출할 때는 `persist=True`를 꼭 줘야 이전 프레임의 ID가 유지됩니다. 통합 실습에서 이 방식을 씁니다.

In [ ]:
def id_color(tid):
    rng = np.random.default_rng(int(tid))
    return tuple(int(c) for c in rng.integers(60, 255, 3))

def run_tracking(src, dst='tracked.mp4', tracker='bytetrack.yaml', trail=25):
    writer = cv2.VideoWriter(dst, cv2.VideoWriter_fourcc(*'mp4v'), CLIP_FPS, (W, H))
    history = defaultdict(list)      # track_id → [(frame, cx, cy), ...]
    for i, r in enumerate(best_model.track(src, stream=True, tracker=tracker,
                                           conf=BEST_CONF, imgsz=CFG['imgsz'], verbose=False)):
        frame = r.orig_img.copy()
        if r.boxes.id is not None:
            for (x1, y1, x2, y2), tid in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.id.int().cpu().tolist()):
                cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
                history[tid].append((i, cx, cy))
                col = id_color(tid)
                cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), col, 2)
                cv2.putText(frame, f'ID {tid}', (int(x1), int(y1) - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.6, col, 2)
                pts = np.array([(int(x), int(y)) for _, x, y in history[tid][-trail:]], np.int32)
                cv2.polylines(frame, [pts], False, col, 2)
        writer.write(frame)
    writer.release()
    return history

track_hist = run_tracking(CLIP)
life = pd.DataFrame([{'track_id': t, 'frames': len(h), 'first_frame': h[0][0], 'last_frame': h[-1][0],
                      'dx': round(float(h[-1][1] - h[0][1]), 1), 'dy': round(float(h[-1][2] - h[0][2]), 1)}
                     for t, h in track_hist.items()]).sort_values('first_frame')
print('부여된 ID 수:', len(life), '| 5프레임 이상 이어진 ID 수:', int((life['frames'] >= 5).sum()))
show_video('tracked.mp4')
life.head(10)

**확인해 보기**
- `dx`(처음 위치 대비 마지막 위치의 x 이동량)의 부호를 보면 상자가 어느 방향으로 움직이는지 알 수 있습니다.
- 몇 프레임만 살다 사라진 짧은 ID들은 무엇일까요? 상자가 가려졌다 다시 보이며 **새 ID를 받는 현상(ID switch)** 은 집계 오차의 주요 원인입니다.

---
## Step 4. 라인 카운팅

화면에 **가상의 기준선**을 긋고, 어떤 ID의 중심점이 이전 프레임에는 선의 한쪽에, 현재 프레임에는 반대쪽에 있으면 '통과'로 셉니다. ID마다 한 번만 세므로 같은 상자를 중복해서 세지 않습니다.

```
트래킹 ID별 중심점 이력 → 선을 넘었는가? → 처음 넘은 ID만 카운트 → 화면 표시 + 로그 기록
```

### 4-1. 이동 방향과 기준선 위치 정하기

In [ ]:
moving = life[(life['frames'] >= 5) & (life['dx'].abs() > W * 0.1)]   # 충분히 오래, 충분히 멀리 움직인 ID만
main_dir = 'right_to_left' if moving['dx'].median() < 0 else 'left_to_right'
print('상자의 주 이동 방향:', main_dir, '| dx 중앙값:', moving['dx'].median())

LINE_X = int(W * 0.5)    # 화면 가운데에 세로 기준선 (필요하면 비율을 바꿔 보세요)
cap = cv2.VideoCapture(CLIP); cap.set(cv2.CAP_PROP_POS_FRAMES, 100); _, preview = cap.read(); cap.release()
cv2.line(preview, (LINE_X, 0), (LINE_X, H), (0, 255, 255), 3)
show_bgr(preview, f'Counting line at x={LINE_X}', size=(4, 7))

In [ ]:
def run_counting(src, dst='counting.mp4', line_x=LINE_X, tracker='bytetrack.yaml'):
    writer = cv2.VideoWriter(dst, cv2.VideoWriter_fourcc(*'mp4v'), CLIP_FPS, (W, H))
    last_x, counted, log = {}, {}, []
    for i, r in enumerate(best_model.track(src, stream=True, tracker=tracker,
                                           conf=BEST_CONF, imgsz=CFG['imgsz'], verbose=False)):
        frame = r.orig_img.copy()
        if r.boxes.id is not None:
            for (x1, y1, x2, y2), tid in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.id.int().cpu().tolist()):
                cx = (x1 + x2) / 2
                prev = last_x.get(tid)
                if prev is not None and tid not in counted:
                    if prev < line_x <= cx:
                        counted[tid] = 'left_to_right'
                    elif prev >= line_x > cx:
                        counted[tid] = 'right_to_left'
                    if tid in counted:
                        log.append({'frame': i, 'time_s': round(i / CLIP_FPS, 2), 'track_id': tid,
                                    'direction': counted[tid]})
                last_x[tid] = cx
                col = (0, 200, 0) if tid in counted else (0, 165, 255)
                cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), col, 2)
                cv2.putText(frame, f'ID {tid}', (int(x1), int(y1) - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.6, col, 2)
        flash = log and log[-1]['frame'] >= i - 2          # 방금 카운트되면 선을 잠깐 빨갛게
        cv2.line(frame, (line_x, 0), (line_x, H), (0, 0, 255) if flash else (0, 255, 255), 3)
        cv2.rectangle(frame, (8, 8), (200, 48), (0, 0, 0), -1)
        cv2.putText(frame, f'COUNT: {len(counted)}', (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2)
        writer.write(frame)
    writer.release()
    return pd.DataFrame(log, columns=['frame', 'time_s', 'track_id', 'direction'])

count_log = run_counting(CLIP)
count_log.to_csv('box_count_log.csv', index=False); save_out('box_count_log.csv')
print('통과한 상자 수:', len(count_log))
show_video('counting.mp4')
count_log

**확인해 보기** — `counting.mp4`를 보면서 선을 넘는 상자를 직접 세어 보세요. 모델의 카운트와 다르다면 어느 장면에서 차이가 났나요? 상자가 붙어서 하나로 잡혔는지, 선 근처에서 ID가 바뀌었는지 확인합니다.

여기까지가 '몇 개 지나갔나'입니다. 현장에서는 여기서 한 걸음 더 나아가 **어떤 크기의 상자가 몇 개 지나갔나**를 알아야 분류 라인과 적재 공간을 계획할 수 있습니다.

---
## Step 5. 크기 분류 — 크기별로 세기

```
마스크 → 크기 측정 (면적, 긴 변, 짧은 변) → S/M/L 기준을 데이터로 정하기
      → 트랙별 크기 확정 (가장자리에서 잘린 프레임 제외, 중앙값) → 크기별 카운트 + 영상 + 로그
```

### 5-1. 마스크로 크기 재기

마스크 윤곽선(`r.masks.xy`)에서 세 가지 값을 구합니다.

| 값 | 계산 | 의미 |
|---|---|---|
| `area_px` | 윤곽선 내부 면적 `cv2.contourArea` | 화면에서 상자가 차지하는 실제 면적 |
| `long_px`, `short_px` | 윤곽선을 가장 작게 감싸는 **회전 사각형** `cv2.minAreaRect`의 두 변 | 놓인 각도와 상관없는 상자의 가로·세로 |

In [ ]:
def measure_mask(poly):
    """마스크 윤곽선 (N,2) → (면적, 긴 변, 짧은 변, 회전 사각형 꼭짓점)"""
    pts = np.asarray(poly, np.float32)
    area = float(cv2.contourArea(pts))
    rect = cv2.minAreaRect(pts)
    (w_, h_) = rect[1]
    return area, float(max(w_, h_)), float(min(w_, h_)), cv2.boxPoints(rect)

# 영상의 한 프레임에서 확인
cap = cv2.VideoCapture(CLIP); cap.set(cv2.CAP_PROP_POS_FRAMES, 100); _, frame = cap.read(); cap.release()
r = best_model(frame, conf=BEST_CONF, imgsz=CFG['imgsz'], verbose=False)[0]
vis = frame.copy()
for b, poly in zip(r.boxes.xyxy.cpu().numpy(), r.masks.xy if r.masks is not None else []):
    if len(poly) < 3:
        continue
    area, long_px, short_px, corners = measure_mask(poly)
    vis = draw_polys(vis, [poly], (0, 255, 0))
    cv2.polylines(vis, [corners.astype(np.int32)], True, (255, 0, 255), 2)
    draw_boxes(vis, [b], (0, 0, 255), thick=1)
    cv2.putText(vis, f'{area:.0f}px2', (int(b[0]), int(b[1]) - 22), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
    cv2.putText(vis, f'{long_px:.0f}x{short_px:.0f}', (int(b[0]), int(b[1]) - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
show_bgr(vis, 'green = mask, magenta = rotated rect, red = box', size=(4, 7))

### 5-2. S/M/L 기준을 데이터로 정하기

기준값을 감으로 정하지 않고 **정답 라벨에서** 정합니다. 같은 카메라(`frame_*` 이미지)의 오른쪽 화면에 있는 상자 다각형 면적을 모두 모아 분포를 보고, 하위 1/3과 상위 1/3을 가르는 값(33%, 67% 분위수)을 S/M/L 경계로 씁니다.

- 학습용 사본에는 회전·확대 변형이 섞여 있어 크기가 달라질 수 있으므로 **변형이 없는 val/test 라벨만** 씁니다.
- 화면 가장자리에 걸려 **일부만 보이는 상자는 제외**합니다. 잘린 상자는 실제보다 작게 재지기 때문입니다.

In [ ]:
EDGE = 4   # 이 픽셀 안쪽으로 가장자리에 닿으면 '잘린 상자'로 봅니다

label_rows = []
for split in ['val', 'test']:
    for img_path in (DATA_ROOT / 'images' / split).glob('frame_*.jpg'):
        _, polys = read_gt(img_path)
        for poly in polys:
            xs, ys = poly[:, 0] - 320, poly[:, 1]          # 오른쪽 화면 기준 좌표 (영상과 같은 좌표계)
            if xs.min() < EDGE or xs.max() > W - EDGE or ys.min() < EDGE or ys.max() > H - EDGE:
                continue                                    # 왼쪽 화면 상자이거나 가장자리에 잘린 상자
            area, long_px, short_px, _ = measure_mask(np.stack([xs, ys], 1))
            label_rows.append({'image': img_path.name, 'area_px': area, 'long_px': long_px, 'short_px': short_px})
label_sizes = pd.DataFrame(label_rows)

q33, q67 = np.percentile(label_sizes['area_px'], [33.3, 66.7])
SIZE_RULES = {'measure': 'mask_area_px', 'small_max': round(float(q33)), 'medium_max': round(float(q67)),
              'edge_margin_px': EDGE, 'line_x': LINE_X, 'conf': BEST_CONF, 'imgsz': CFG['imgsz']}
json.dump(SIZE_RULES, open('size_rules.json', 'w'), indent=2); save_out('size_rules.json')
print(f'라벨 상자 {len(label_sizes)}개로 기준을 정했습니다:', SIZE_RULES)

plt.figure(figsize=(8, 3.2))
plt.hist(label_sizes['area_px'], bins=40, color='tab:gray')
for v, name in [(q33, 'S | M'), (q67, 'M | L')]:
    plt.axvline(v, c='red', ls='--'); plt.text(v, plt.ylim()[1] * 0.9, f' {name}\n {v:.0f}', color='red')
plt.xlabel('mask area (px^2)'); plt.ylabel('labels'); plt.title('Package size distribution (ground truth)')
plt.show()

In [ ]:
def size_class(area, rules=SIZE_RULES):
    if area <= rules['small_max']:
        return 'S'
    if area <= rules['medium_max']:
        return 'M'
    return 'L'

SIZE_COLOR = {'S': (255, 200, 0), 'M': (0, 200, 255), 'L': (60, 60, 255)}   # BGR: 하늘색, 주황, 빨강
print([size_class(a) for a in [1000, 5000, 15000]])

### 5-3. 관측 기록 만들기 — 한 번 추적하고 표로 남기기

트래킹을 한 번만 돌리면서 **프레임마다, 상자마다** 위치와 크기를 표 한 줄로 기록합니다. 이후의 집계·분류·영상 만들기는 모두 이 표로 합니다.
모델을 다시 돌리지 않고도 기준을 바꿔 가며 몇 번이고 다시 집계할 수 있습니다.

In [ ]:
def collect_observations(src, tracker='bytetrack.yaml'):
    obs, polys = [], {}
    for i, r in enumerate(best_model.track(src, stream=True, tracker=tracker,
                                           conf=BEST_CONF, imgsz=CFG['imgsz'], verbose=False)):
        if r.boxes.id is None or r.masks is None:
            continue
        for b, tid, poly in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.id.int().cpu().tolist(), r.masks.xy):
            if len(poly) < 3:
                continue
            area, long_px, short_px, _ = measure_mask(poly)
            x1, y1, x2, y2 = b
            obs.append({'frame': i, 'track_id': tid, 'cx': (x1 + x2) / 2, 'cy': (y1 + y2) / 2,
                        'area_px': area, 'long_px': long_px, 'short_px': short_px,
                        'at_edge': bool(x1 < EDGE or y1 < EDGE or x2 > W - EDGE or y2 > H - EDGE)})
            polys[(i, tid)] = poly
    return pd.DataFrame(obs), polys

obs, obs_polys = collect_observations(CLIP)
print('관측 행 수:', len(obs), '| 트랙 수:', obs['track_id'].nunique(),
      '| 가장자리에 걸린 관측 비율:', round(obs['at_edge'].mean(), 2))
obs.round(1).head()

In [ ]:
# 한 상자의 면적이 시간에 따라 어떻게 재지는지 확인
tid_example = obs.groupby('track_id').size().idxmax()
one = obs[obs['track_id'] == tid_example]
plt.figure(figsize=(8, 3))
plt.plot(one['frame'], one['area_px'], '-', c='gray', lw=1)
plt.scatter(one['frame'], one['area_px'], c=np.where(one['at_edge'], 'red', 'tab:blue'), s=12)
plt.xlabel('frame'); plt.ylabel('mask area (px^2)')
plt.title(f'track {tid_example}: red = touching frame edge (excluded)')
plt.grid(alpha=.3); plt.show()

상자가 화면에 들어오거나 나갈 때(빨간 점)는 일부만 보여 면적이 작게 재집니다. 그래서 **가장자리에 걸리지 않은 관측만 모아 중앙값**으로 그 상자의 크기를 정합니다. 평균 대신 중앙값을 쓰면 마스크가 순간적으로 튀는 값에도 흔들리지 않습니다.

### 5-4. 크기별로 세기

In [ ]:
def summarize_tracks(obs, line_x=LINE_X, fps=CLIP_FPS, rules=SIZE_RULES):
    rows = []
    for tid, g in obs.sort_values('frame').groupby('track_id'):
        prev = g['cx'].shift()
        crossed = g[(prev < line_x) & (g['cx'] >= line_x) | (prev >= line_x) & (g['cx'] < line_x)]
        if crossed.empty:
            continue                                     # 기준선을 넘지 않은 트랙은 집계하지 않음
        first = crossed.iloc[0]
        direction = 'left_to_right' if first['cx'] >= line_x else 'right_to_left'
        full = g[~g['at_edge']]
        use = full if len(full) >= 3 else g             # 온전한 관측이 3개 미만이면 전체 사용
        area = float(use['area_px'].median())
        rows.append({'track_id': tid, 'cross_frame': int(first['frame']),
                     'time_s': round(first['frame'] / fps, 2), 'direction': direction,
                     'size_class': size_class(area, rules), 'area_px': round(area),
                     'long_px': round(float(use['long_px'].median())),
                     'short_px': round(float(use['short_px'].median())),
                     'n_obs': len(use), 'edge_only': len(full) < 3})
    cols = ['track_id', 'cross_frame', 'time_s', 'direction', 'size_class', 'area_px',
            'long_px', 'short_px', 'n_obs', 'edge_only']
    return pd.DataFrame(rows, columns=cols).sort_values('cross_frame').reset_index(drop=True)

size_log = summarize_tracks(obs)
size_log.to_csv('box_size_log.csv', index=False); save_out('box_size_log.csv')

size_counts = size_log['size_class'].value_counts().reindex(['S', 'M', 'L'], fill_value=0)
print('크기별 통과 수:', size_counts.to_dict(), '| 합계:', int(size_counts.sum()))
print('Step 4 카운트와 같은가:', int(size_counts.sum()) == len(count_log))
size_log

Step 4의 카운트와 합계가 같아야 합니다. 같은 모델·같은 트래커·같은 기준선을 썼기 때문입니다. 다르다면 `collect_observations()`에서 마스크가 없는 상자를 건너뛴 경우인지 확인해 보세요.

### 5-5. 결과 영상과 요약 그래프

관측 표와 집계 결과를 이용해 영상을 다시 그립니다. 상자는 크기별 색으로 칠하고, 기준선을 넘는 순간 오른쪽 위의 S/M/L 카운터가 올라갑니다.

In [ ]:
def render_size_video(src, obs_polys, size_log, dst='size_counting.mp4', line_x=LINE_X):
    cls_of = dict(zip(size_log['track_id'], size_log['size_class']))
    cross_at = dict(zip(size_log['track_id'], size_log['cross_frame']))
    by_frame = defaultdict(list)
    for (i, tid), poly in obs_polys.items():
        by_frame[i].append((tid, poly))
    cap = cv2.VideoCapture(src)
    writer = cv2.VideoWriter(dst, cv2.VideoWriter_fourcc(*'mp4v'), CLIP_FPS, (W, H))
    i = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        overlay = frame.copy()
        for tid, poly in by_frame[i]:
            c = cls_of.get(tid)
            col = SIZE_COLOR[c] if c else (180, 180, 180)          # 집계 대상이 아닌 상자는 회색
            cv2.fillPoly(overlay, [poly.astype(np.int32)], col)
            x, y = poly[:, 0].min(), poly[:, 1].min()
            label = f'{c} #{tid}' if c else f'#{tid}'
            cv2.putText(frame, label, (int(x), int(max(14, y - 4))), cv2.FONT_HERSHEY_SIMPLEX, 0.55, col, 2)
        frame = cv2.addWeighted(overlay, 0.45, frame, 0.55, 0)
        cv2.line(frame, (line_x, 0), (line_x, H), (0, 255, 255), 2)
        done = [cls_of[t] for t, f in cross_at.items() if f <= i]
        cv2.rectangle(frame, (6, 6), (W - 6, 40), (0, 0, 0), -1)
        for k, s in enumerate(['S', 'M', 'L']):
            cv2.putText(frame, f'{s}:{done.count(s)}', (14 + k * 72, 32), cv2.FONT_HERSHEY_SIMPLEX, 0.75, SIZE_COLOR[s], 2)
        cv2.putText(frame, f'={len(done)}', (14 + 3 * 72, 32), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (255, 255, 255), 2)
        writer.write(frame)
        i += 1
    cap.release(); writer.release()
    return dst

render_size_video(CLIP, obs_polys, size_log)
save_out('size_counting.mp4')
show_video('size_counting.mp4')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4), gridspec_kw={'width_ratios': [1, 2]})
bar_colors = ['#00c8ff', '#ffc800', '#ff3c3c']
axes[0].bar(['S', 'M', 'L'], size_counts.values, color=bar_colors)
axes[0].set_title('Packages by size'); axes[0].set_ylabel('count')
for s, c in zip(['S', 'M', 'L'], bar_colors):
    d = size_log[size_log['size_class'] == s]
    axes[1].scatter(d['time_s'], d['area_px'], s=60, color=c, label=s, edgecolor='k')
axes[1].axhline(SIZE_RULES['small_max'], ls='--', c='gray', lw=1)
axes[1].axhline(SIZE_RULES['medium_max'], ls='--', c='gray', lw=1)
axes[1].set_xlabel('time crossing the line (s)'); axes[1].set_ylabel('mask area (px^2)')
axes[1].set_title('Each counted package'); axes[1].legend(); axes[1].grid(alpha=.3)
plt.tight_layout(); plt.savefig('size_summary.png', dpi=120); save_out('size_summary.png'); plt.show()

결과 파일 (모두 `SAVE_DIR`에도 저장됩니다)

| 파일 | 내용 |
|---|---|
| `box_seg_best.pt` | 파인튜닝한 상자 세그멘테이션 모델 |
| `conveyor_clip.mp4` | 실습용 컨베이어 영상 |
| `box_count_log.csv` | 통과 기록 (`frame, time_s, track_id, direction`) |
| `size_rules.json` | S/M/L 기준값, 기준선 위치, 임계값 |
| `box_size_log.csv` | 크기별 통과 기록 (`track_id, cross_frame, time_s, direction, size_class, area_px, long_px, short_px, n_obs, edge_only`) |
| `size_counting.mp4` | 크기별 색과 S/M/L 카운터가 표시된 영상 |
| `size_summary.png` | 크기별 개수와 상자별 면적 그래프 |
| `experiments.csv` | 실험별 성능 비교 |

`box_seg_best.pt`, `conveyor_clip.mp4`, `size_rules.json`은 통합 실습에서 다시 사용합니다.

---
## 도전 과제

1. **긴 변 기준으로 분류** — 면적 대신 `long_px`(긴 변)로 S/M/L을 나누면 결과가 어떻게 달라지나요? 납작하고 긴 상자는 어느 쪽 기준에서 더 합리적으로 분류되나요?
2. **원근 보정** — 카메라에서 먼 쪽(화면 위쪽)의 상자는 작게 보입니다. 기준선 위치(세로)에 따라 면적을 보정하는 방법을 생각해 보세요. 힌트: 라벨 면적과 `cy`(세로 위치)의 관계를 그래프로 그려 봅니다.
3. **실시간으로 결정하기** — 지금은 영상을 다 본 뒤 크기를 정합니다. 기준선을 넘는 **그 순간까지** 모인 관측만으로 크기를 정하도록 `summarize_tracks()`를 바꿔 보세요. 결과가 얼마나 달라지나요?
4. **트래커 비교** — `tracker='botsort.yaml'`로 바꿔 ID 수와 크기별 카운트가 어떻게 달라지는지 표로 정리해 보세요.

## 정리

- 사전학습 모델은 출발점일 뿐, **우리 대상은 우리 데이터로 가르쳐야** 합니다.
- 크기를 재려면 박스가 아니라 **마스크**가 필요합니다. 비스듬한 상자의 박스는 실제보다 훨씬 큽니다.
- 기준값은 감이 아니라 **정답 데이터의 분포**로 정하고, 파일(`size_rules.json`)로 남겨야 다른 시스템에서도 같은 기준을 씁니다.
- 영상 분석은 '추적해서 **관측 표로 남기고** → 표로 집계'하는 구조로 만들면, 기준을 바꿔도 모델을 다시 돌릴 필요가 없습니다.